# 05b — Feature Engineering on top of DWS Model
Loads your saved `train_matched` / `test_matched`, adds domain features, trains same CatBoost.

**Available columns**: pet, nir, green, swir16, swir22, NDMI, MNDWI, Lat/Lon, Sample Date, dws_*

In [6]:
!pip install catboost

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from catboost import CatBoostRegressor
import warnings, time
warnings.filterwarnings('ignore')

BASE_DIR = 'data'  # <<< CHANGE
TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
EPS = 1e-6

In [8]:
# Load your saved matched files
train_matched = pd.read_csv(f'{BASE_DIR}/train_ALL+reliability.csv')
test_matched = pd.read_csv(f'{BASE_DIR}/test_ALL+reliability.csv')
print(f"Train: {train_matched.shape}, Test: {test_matched.shape}")
print(f"Columns: {list(train_matched.columns)}")

Train: (9319, 78), Test: (200, 77)
Columns: ['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'pet', '_merge_terra', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', '_merge_landsat', 'Impute_Method', 'geometry', 'STAT_ID', 'sc', 'ss', 'su', 'mt', 'va', 'vb', 'vi', 'pa', 'pb', 'pi', 'GLC_Artificial', 'GLC_Managed', 'GLC_Water', 'GLC_Aquatic_Veg', 'GLC_PERC_COV', 'Popdens_00', 'Soil_pH', 'SOC', 'Soil_wetness', 'dist_m', 'dist_km', 'Latitude_glorich', 'Longitude_glorich', 'date', 'Alkalinity', 'Cl', 'DIP', 'SO4', 'SpecCond25C', 'pH', 'Alkalinity_reliability', 'Cl_reliability', 'DIP_reliability', 'SO4_reliability', 'SpecCond25C_reliability', 'pH_reliability', 'date_diff_days', 'dws_1st', 'dws_1st_dist', 'dws_TAL', 'dws_EC', 'dws_pH', 'dws_Ca', 'dws_Mg', 'dws_Na', 'dws_Cl', 'dws_SO4', 'dws_days_diff', 'dws_dist_km', 'dws_P_modified', 'P_modified_same', 'dws_P_reliability', 'month', 'day_of_year', 'year', 'month_sin',

In [9]:
# ==========================================
# FEATURE ENGINEERING
# ==========================================
def add_features(df):
    d = df.copy()
    
    # =========================
    # TIME FEATURES (from Sample Date)
    # =========================
    dt = pd.to_datetime(d['Sample Date'], dayfirst=True, format='mixed')
    d['month'] = dt.dt.month
    d['year'] = dt.dt.year
    d['day_of_year'] = dt.dt.dayofyear
    d['month_sin'] = np.sin(2 * np.pi * d['month'] / 12)
    d['month_cos'] = np.cos(2 * np.pi * d['month'] / 12)
    d['doy_sin'] = np.sin(2 * np.pi * d['day_of_year'] / 365)
    d['doy_cos'] = np.cos(2 * np.pi * d['day_of_year'] / 365)
    # SA wet season: Oct-Mar
    d['wet_season'] = d['month'].isin([10,11,12,1,2,3]).astype(int)
    
    # =========================
    # SPECTRAL BAND RATIOS
    # =========================
    if all(c in d.columns for c in ['nir','green','swir16','swir22']):
        d['nir_green_ratio'] = d['nir'] / (d['green'] + EPS)
        d['swir16_nir_ratio'] = d['swir16'] / (d['nir'] + EPS)
        d['swir22_nir_ratio'] = d['swir22'] / (d['nir'] + EPS)
        d['swir16_green_ratio'] = d['swir16'] / (d['green'] + EPS)
        d['nir_minus_green'] = d['nir'] - d['green']
        d['swir_diff'] = d['swir16'] - d['swir22']
    
    # =========================
    # EC-SPECIFIC FEATURES
    # =========================
    # EC = dissolved ions → driven by evaporation, geology, seasonal concentration
    
    # 1. Evapoconcentration (PET × season)
    if 'pet' in d.columns:
        d['ec_pet_x_dry'] = d['pet'] * (1 - d['wet_season'])
        d['ec_pet_x_wet'] = d['pet'] * d['wet_season']
        d['ec_pet_x_msin'] = d['pet'] * d['month_sin']
        d['ec_pet_x_mcos'] = d['pet'] * d['month_cos']
    
    # 2. Salinity indices from SWIR (salinity mapping literature)
    if all(c in d.columns for c in ['swir16','swir22','nir','green']):
        d['ec_salinity_idx'] = np.sqrt(np.abs(d['swir16'] * d['swir22']))
        d['ec_ndsi'] = (d['swir16'] - d['swir22']) / (d['swir16'] + d['swir22'] + EPS)
        d['ec_brightness'] = np.sqrt(d['green']**2 + d['nir']**2 + d['swir16']**2)
    
    # 3. DWS ion composites (EC is literally sum of dissolved ions)
    ion_cols = [c for c in ['dws_Ca','dws_Mg','dws_Na','dws_Cl','dws_SO4'] if c in d.columns]
    if len(ion_cols) >= 2:
        d['ec_total_ions'] = d[ion_cols].sum(axis=1)
    if all(c in d.columns for c in ['dws_Cl','dws_SO4']):
        d['ec_cl_so4_ratio'] = d['dws_Cl'] / (d['dws_SO4'] + EPS)
    if all(c in d.columns for c in ['dws_Ca','dws_Mg','dws_Na']):
        d['ec_camg_na_ratio'] = (d['dws_Ca'] + d['dws_Mg']) / (d['dws_Na'] + EPS)
    
    # 4. Distance-decayed DWS EC
    if all(c in d.columns for c in ['dws_EC','dws_dist_km','dws_days_diff']):
        d['ec_dws_decay'] = d['dws_EC'] / (1 + d['dws_dist_km']*0.1 + np.abs(d['dws_days_diff'])*0.01)
    
    # =========================
    # DRP-SPECIFIC FEATURES
    # =========================
    # DRP = phosphorus → driven by ag runoff, sewage, high-flow flushing
    
    # 1. Seasonal flush indicators
    # DRP is ~4x higher in summer (SA wet season) — fertilizer + sewage overflow
    d['drp_peak_flush'] = d['month'].isin([12,1,2,3]).astype(int)
    d['drp_first_flush'] = d['month'].isin([10,11]).astype(int)
    
    # 2. Runoff proxies (NDMI/MNDWI × season)
    if 'NDMI' in d.columns:
        d['drp_ndmi_x_wet'] = d['NDMI'] * d['wet_season']
    if 'MNDWI' in d.columns:
        d['drp_mndwi_x_wet'] = d['MNDWI'] * d['wet_season']
    if all(c in d.columns for c in ['NDMI','MNDWI']):
        d['drp_water_signal'] = d['MNDWI'] * d['NDMI']  # both high = active flow
    
    # 3. Algal bloom spectral proxies
    if all(c in d.columns for c in ['nir','green']):
        d['drp_chl_proxy'] = d['nir'] / (d['green'] + EPS)  # chlorophyll-a proxy
        d['drp_green_dom'] = d['green'] / (d['nir'] + d['green'] + EPS)  # eutrophic water
    if all(c in d.columns for c in ['nir','swir16']):
        d['drp_turbidity'] = (d['nir'] - d['swir16']) / (d['nir'] + d['swir16'] + EPS)
    
    # 4. PET as inverse runoff proxy
    if 'pet' in d.columns:
        d['drp_inv_pet'] = 1.0 / (d['pet'] + EPS)  # low PET = cool/wet = more runoff
        d['drp_pet_x_wet'] = d['pet'] * d['wet_season']
    
    # 5. DWS phosphorus decay-weighted
    if all(c in d.columns for c in ['dws_PO4_P','dws_dist_km','dws_days_diff']):
        d['drp_dws_decay'] = d['dws_PO4_P'] / (1 + d['dws_dist_km']*0.1 + np.abs(d['dws_days_diff'])*0.01)
    
    # 6. DWS PO4 × season (P runoff is seasonal)
    if 'dws_PO4_P' in d.columns:
        d['drp_po4_x_wet'] = d['dws_PO4_P'] * d['wet_season']
    
    # =========================
    # SPATIAL FEATURES
    # =========================
    if all(c in d.columns for c in ['Latitude','Longitude']):
        d['lat_lon_interact'] = d['Latitude'] * d['Longitude']
    
    return d

t0 = time.time()
train_feat = add_features(train_matched)
test_feat = add_features(test_matched)
print(f"Feature engineering done in {time.time()-t0:.1f}s")
print(f"Train: {train_feat.shape}, Test: {test_feat.shape}")

# Show new features
new_cols = [c for c in train_feat.columns if c not in train_matched.columns]
print(f"\nNew features ({len(new_cols)}):")
for c in new_cols:
    print(f"  {c}")

Feature engineering done in 0.1s
Train: (9319, 106), Test: (200, 105)

New features (28):
  nir_green_ratio
  swir16_nir_ratio
  swir22_nir_ratio
  swir16_green_ratio
  nir_minus_green
  swir_diff
  ec_pet_x_dry
  ec_pet_x_wet
  ec_pet_x_msin
  ec_pet_x_mcos
  ec_salinity_idx
  ec_ndsi
  ec_brightness
  ec_total_ions
  ec_cl_so4_ratio
  ec_camg_na_ratio
  ec_dws_decay
  drp_peak_flush
  drp_first_flush
  drp_ndmi_x_wet
  drp_mndwi_x_wet
  drp_water_signal
  drp_chl_proxy
  drp_green_dom
  drp_turbidity
  drp_inv_pet
  drp_pet_x_wet
  lat_lon_interact


In [10]:
# ==========================================
# SPATIAL CV SETUP (same as your 05 notebook)
# ==========================================
stations = train_feat.groupby(['Latitude', 'Longitude']).size().reset_index().rename(columns={0: 'n'})
km = KMeans(n_clusters=5, random_state=42, n_init=10)
stations['spatial_cluster'] = km.fit_predict(stations[['Latitude', 'Longitude']])
train_feat = train_feat.merge(stations[['Latitude', 'Longitude', 'spatial_cluster']],
                               on=['Latitude', 'Longitude'], how='left')

# Same drop_cols as your 05 notebook
drop_cols = (
    TARGETS +
    ['Sample Date', 'spatial_cluster',
     'geometry', '_merge_terra', '_merge_landsat',
     'Latitude_glorich', 'Longitude_glorich', 'date',
     'dws_1st',  # station ID string
     'day_of_year',  # redundant with sin/cos
    ]
)

feature_cols = [c for c in train_feat.columns if c not in drop_cols
                and c in test_feat.columns]  # only cols that exist in both

# Remove any non-numeric except Impute_Method
for c in feature_cols[:]:
    if train_feat[c].dtype == 'object' and c != 'Impute_Method':
        feature_cols.remove(c)

print(f"Total features: {len(feature_cols)}")
print(f"EC-specific: {[c for c in feature_cols if c.startswith('ec_')]}")
print(f"DRP-specific: {[c for c in feature_cols if c.startswith('drp_')]}")

Total features: 93
EC-specific: ['ec_pet_x_dry', 'ec_pet_x_wet', 'ec_pet_x_msin', 'ec_pet_x_mcos', 'ec_salinity_idx', 'ec_ndsi', 'ec_brightness', 'ec_total_ions', 'ec_cl_so4_ratio', 'ec_camg_na_ratio', 'ec_dws_decay']
DRP-specific: ['drp_peak_flush', 'drp_first_flush', 'drp_ndmi_x_wet', 'drp_mndwi_x_wet', 'drp_water_signal', 'drp_chl_proxy', 'drp_green_dom', 'drp_turbidity', 'drp_inv_pet', 'drp_pet_x_wet']


In [11]:
# ==========================================
# SPATIAL CV — same CatBoost params as your 05
# ==========================================
def evaluate_spatial_cv(X, y, train_df, target_name, transform=None, n_splits=5):
    y_work = np.log1p(y) if transform == 'log1p' else y.copy()
    gkf = GroupKFold(n_splits=n_splits)
    groups = train_df['spatial_cluster'].values
    fold_scores = []

    cat_idx = [feature_cols.index(c) for c in ['Impute_Method'] if c in feature_cols]

    for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y_work.iloc[tr_idx], y_work.iloc[val_idx]

        model = CatBoostRegressor(
            iterations=800, learning_rate=0.05, depth=6,
            l2_leaf_reg=3, verbose=0, random_state=42,
            cat_features=cat_idx if cat_idx else None,
        )
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        if transform == 'log1p':
            preds_raw = np.expm1(preds)
            y_val_raw = y.iloc[val_idx]
        else:
            preds_raw = preds
            y_val_raw = y_val

        score = r2_score(y_val_raw, preds_raw)
        fold_scores.append(score)
        print(f"  Fold {fold_idx}: R²={score:.4f}")

    mean_r2 = np.mean(fold_scores)
    print(f"  → {target_name}: Mean R²={mean_r2:.4f} ± {np.std(fold_scores):.4f}")
    return mean_r2

X = train_feat[feature_cols].copy()
results = {}
for target in TARGETS:
    y = train_feat[target].copy()
    transform = 'log1p' if target == 'Dissolved Reactive Phosphorus' else None
    print(f"\n{'='*50}")
    print(f"{target} {'(log1p)' if transform else ''}")
    print(f"{'='*50}")
    results[target] = evaluate_spatial_cv(X, y, train_feat, target, transform)

mean_all = np.mean(list(results.values()))
print(f"\n{'='*50}")
print(f"MEAN R² = {mean_all:.4f}")
print(f"{'='*50}")


Total Alkalinity 
  Fold 0: R²=0.9793
  Fold 1: R²=0.9715
  Fold 2: R²=0.9885
  Fold 3: R²=0.9644
  Fold 4: R²=0.9902
  → Total Alkalinity: Mean R²=0.9788 ± 0.0098

Electrical Conductance 
  Fold 0: R²=0.9854
  Fold 1: R²=0.9343
  Fold 2: R²=0.9146
  Fold 3: R²=0.7807
  Fold 4: R²=0.9645
  → Electrical Conductance: Mean R²=0.9159 ± 0.0719

Dissolved Reactive Phosphorus (log1p)
  Fold 0: R²=-0.0017
  Fold 1: R²=0.7838
  Fold 2: R²=0.5582
  Fold 3: R²=0.7373
  Fold 4: R²=0.8295
  → Dissolved Reactive Phosphorus: Mean R²=0.5814 ± 0.3057

MEAN R² = 0.8254


In [12]:
# ==========================================
# TRAIN FINAL + SUBMISSION
# ==========================================
X_train = train_feat[feature_cols].copy()
X_test = test_feat[feature_cols].copy()

cat_idx = [feature_cols.index(c) for c in ['Impute_Method'] if c in feature_cols]
test_preds = {}

for target in TARGETS:
    y = train_feat[target].copy()
    transform = 'log1p' if target == 'Dissolved Reactive Phosphorus' else None
    y_work = np.log1p(y) if transform == 'log1p' else y

    model = CatBoostRegressor(
        iterations=800, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, verbose=100, random_state=42,
        cat_features=cat_idx if cat_idx else None,
    )
    model.fit(X_train, y_work)

    preds = model.predict(X_test)
    if transform == 'log1p':
        preds = np.expm1(preds)
    preds = np.clip(preds, 0, None)
    test_preds[target] = preds
    print(f"{target}: [{preds.min():.2f}, {preds.max():.2f}], mean={preds.mean():.2f}")

# === HYBRID SUBMISSION (same as your 05) ===
submission = pd.DataFrame({
    'Latitude': test_feat['Latitude'],
    'Longitude': test_feat['Longitude'],
    'Sample Date': test_matched['Sample Date'],
})

dws_tal = test_feat['dws_TAL'].values
submission['Total Alkalinity'] = np.where(
    np.isfinite(dws_tal), dws_tal, test_preds['Total Alkalinity']
)
submission['Electrical Conductance'] = test_preds['Electrical Conductance']
submission['Dissolved Reactive Phosphorus'] = test_preds['Dissolved Reactive Phosphorus']

submission.to_csv(f'{BASE_DIR}/submission_dws_feat_eng.csv', index=False)
print(f"\nSubmission saved.")
print(submission[TARGETS].describe())

0:	learn: 71.3608008	total: 111ms	remaining: 1m 29s
100:	learn: 5.3721948	total: 15.8s	remaining: 1m 49s
200:	learn: 4.1712920	total: 30.8s	remaining: 1m 31s
300:	learn: 3.6944883	total: 47.5s	remaining: 1m 18s
400:	learn: 3.3726173	total: 1m 2s	remaining: 1m 2s
500:	learn: 3.1292089	total: 1m 17s	remaining: 46.1s
600:	learn: 2.9390244	total: 1m 32s	remaining: 30.6s
700:	learn: 2.7606232	total: 1m 46s	remaining: 15s
799:	learn: 2.6035355	total: 2m	remaining: 0us
Total Alkalinity: [16.79, 358.73], mean=120.83
0:	learn: 327.4976830	total: 118ms	remaining: 1m 33s
100:	learn: 50.3319398	total: 16s	remaining: 1m 50s
200:	learn: 36.8110812	total: 31.3s	remaining: 1m 33s
300:	learn: 30.4804403	total: 46.1s	remaining: 1m 16s
400:	learn: 26.2626116	total: 1m	remaining: 1m
500:	learn: 23.0843390	total: 1m 15s	remaining: 45.2s
600:	learn: 20.4787886	total: 1m 30s	remaining: 30s
700:	learn: 18.4212383	total: 1m 46s	remaining: 15s
799:	learn: 16.8018390	total: 2m	remaining: 0us
Electrical Conductan

In [13]:
# ==========================================
# QUICK IMPORTANCE CHECK — did new features help?
# ==========================================
for target in ['Electrical Conductance', 'Dissolved Reactive Phosphorus']:
    y = train_feat[target].copy()
    transform = 'log1p' if target == 'Dissolved Reactive Phosphorus' else None
    y_work = np.log1p(y) if transform == 'log1p' else y
    
    model = CatBoostRegressor(
        iterations=800, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, verbose=0, random_state=42,
        cat_features=cat_idx if cat_idx else None,
    )
    model.fit(X_train, y_work)
    
    fi = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': model.get_feature_importance()
    }).sort_values('Importance', ascending=False)
    
    prefix = 'ec_' if 'Conductance' in target else 'drp_'
    print(f"\n{'='*40}")
    print(f"{target} — Top 15")
    print(f"{'='*40}")
    for _, row in fi.head(15).iterrows():
        marker = ' ★' if row['Feature'].startswith(prefix) else ''
        print(f"  {row['Importance']:6.1f}  {row['Feature']}{marker}")
    
    new_fi = fi[fi['Feature'].str.startswith(prefix)]
    print(f"  Target-specific gain: {new_fi['Importance'].sum():.1f}% total")


Electrical Conductance — Top 15
    38.3  ec_dws_decay ★
    30.9  dws_EC
     4.3  ec_total_ions ★
     1.7  Soil_pH
     1.5  SpecCond25C
     1.4  dws_Mg
     1.3  dws_TAL
     1.2  dws_Na
     1.1  dws_Cl
     1.0  dws_Ca
     1.0  SOC
     0.9  dws_dist_km
     0.9  DIP
     0.9  Cl
     0.9  dws_1st_dist
  Target-specific gain: 43.4% total

Dissolved Reactive Phosphorus — Top 15
    49.5  dws_P_modified
     9.9  year
     6.4  dws_P_reliability
     2.6  DIP
     1.4  vi
     1.3  dws_dist_km
     1.3  Latitude
     1.2  mt
     1.1  pH
     1.1  dws_Na
     1.0  dws_1st_dist
     0.9  pet
     0.9  GLC_Water
     0.8  dws_Ca
     0.8  Popdens_00
  Target-specific gain: 1.4% total
